# Pydantic: Structured, Validated Data for AI Applications

Pydantic models define the shape of application data and validate values when instances are created. Ordinary Python classes can implement validation, but Python does not add runtime validation automatically just because a class has attributes or type annotations; Pydantic provides that behavior through `BaseModel`, field constraints, and clear validation errors.

Structured data makes application code more predictable and gives language models an explicit schema to target. In LangChain and LangGraph, typed schemas are useful for structured model outputs, tool inputs, and application state. Pydantic is not required to build every agent or chatbot, and schema validation cannot guarantee that an LLM's answer is factually correct, but it is a valuable foundation for reliable AI workflows.

This notebook builds from a validated product model to nested orders and carts, then shows a reusable collection type with a safe empty default.

## Define a Data Shape and Validate Its Fields

`Product` declares the fields an application expects. `Field` adds constraints beyond the basic Python types, so invalid values are rejected when Pydantic creates the model. A valid instance then exposes predictable attributes to the rest of the program.

In [2]:
# BaseModel provides runtime validation; Field adds constraints to each value.
from pydantic import BaseModel, Field

In [3]:
# Define a product schema with length limits and a positive-price constraint.
class Product(BaseModel):
    title:str = Field(max_length=20)
    desc:str = Field(min_length=20, max_length=100)
    price:int = Field(gt=0)

new_product = Product(
    title="iPhone 17 Pro",
    desc = "The best iPhone Apple has ever released",
    price = 1999
)

print(new_product.title)
print(new_product.desc)
print(new_product.price)

iPhone 17 Pro
The best iPhone Apple has ever released
1999


## See What Happens When Data Is Invalid

The next example intentionally violates all three field constraints. Pydantic reports the validation failures instead of silently constructing a valid-looking `Product`. This checks data shape and constraints—not whether a product description or an LLM-generated claim is true.

In [4]:
# Each supplied value violates a Product constraint, so Pydantic raises ValidationError.
bad_product = Product(
    title="This product will change your life with its new formula",
    desc = "magic",
    price = -2
)


ValidationError: 3 validation errors for Product
title
  String should have at most 20 characters [type=string_too_long, input_value='This product will change...fe with its new formula', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/string_too_long
desc
  String should have at least 20 characters [type=string_too_short, input_value='magic', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/string_too_short
price
  Input should be greater than 0 [type=greater_than, input_value=-2, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than

## Compose Models into Nested Application Data

An order can contain a validated `Product` rather than a loosely structured dictionary. Nesting models lets an application keep related values organized while reusing the same field rules.

In [8]:
# Reuse the Product schema inside an order instead of duplicating its fields.
class Orders(BaseModel):
    id:str = Field(min_length=2, max_length=2)
    product: Product

new_order = Orders(id = "A1", product = new_product)
print(new_order.id)
print(new_order.product)
print(new_order.product.title)
print(new_order.product.desc)
print(new_order.product.price)

A1
title='iPhone 17 Pro' desc='The best iPhone Apple has ever released' price=1999
iPhone 17 Pro
The best iPhone Apple has ever released
1999


## Represent Collections of Validated Models

A cart contains a session identifier and a list of `Product` instances. The list annotation communicates the expected item type, and Pydantic validates each nested product as part of the cart.

In [11]:
# A cart groups multiple validated products under one session identifier.
from typing import List

class Cart(BaseModel):
    session_id: int
    products: List[Product]

previous_product = new_product
new_product = Product(
    title="iPhone 18 Pro",
    desc = "The best iPhone Apple has ever released",
    price = 2199
)

cart = Cart(session_id=1, products=[previous_product, new_product])
print(cart)

session_id=1 products=[Product(title='iPhone 18 Pro', desc='The best iPhone Apple has ever released', price=2199), Product(title='iPhone 18 Pro', desc='The best iPhone Apple has ever released', price=2199)]


The model fields can be used directly in application logic. This loop demonstrates iterating over the cart's products and reading their validated attributes.

In [17]:
# Iterate over the structured collection and access each product's fields.
for i, prod in enumerate(cart.products):
    print("----------------------------------------")
    print("Product", i + 1)
    print(prod.title)
    print(prod.desc)
    print(prod.price)
    print("----------------------------------------")

----------------------------------------
Product 1
iPhone 18 Pro
The best iPhone Apple has ever released
2199
----------------------------------------
----------------------------------------
Product 2
iPhone 18 Pro
The best iPhone Apple has ever released
2199
----------------------------------------


## Give Optional Collections a Safe Default

A cart may start with no products. `Annotated` lets us package the product-list type and its Pydantic field default under one reusable alias, so the model can omit `products` when creating an empty cart.

In [21]:
# Annotate a reusable list-of-products type with an empty default.
from typing import Annotated
commonValidator = Annotated[List[Product], Field(default=[])]

In [22]:
# Reuse the annotated type so products default to an empty list when omitted.
class Cart(BaseModel):
    session_id: int
    products: commonValidator
cart = Cart(session_id=2)
print(cart)

session_id=2 products=[]
